> 请点击获取[课程 PPT 内容](https://www.canva.cn/design/DAG9KgDFL-g/JuY78K8O2tquhxpDmVhlSA/view?utm_content=DAG9KgDFL-g&utm_campaign=designshare&utm_medium=link2&utm_source=uniquelinks&utlId=h89f16422ac)。


# 1. 环境配置

## 1.1 python 环境准备

In [ ]:
! pip install "langgraph-cli[inmem]" openai==2.11.0 dashscope==1.25.4 langchain-classic==1.0.0 langchain==1.2.0 langchain-community==0.4.1 langchain-openai==1.1.6 arxiv==2.3.1

## 1.2 大模型密钥准备

请根据第一章内容获取相关平台的 API KEY，如若未在系统变量中填入，请将 API_KEY 信息写入以下代码（若已设置请忽略）：

In [ ]:
import os

# os.environ["OPENAI_API_KEY"] = "sk-xxxxxxxx"
# os.environ["DASHSCOPE_API_KEY"] = "sk-yyyyyyyy"

# 2. Subagents 模式
## 2.1 简介

Subagents 是最传统的多智能体方法，其主要使用方式就是利用主智能体指挥划分好的子智能体来进行任务的完成。

假如使用 Subagents 的方式构建多智能体系统，通常分为以下几步：
- 创建底层工具
- 构建子智能体（连接底层工具）
- 将子代理包装成工具（给 Supervisor 使用）
- 构建 Supervisor Agent（主智能体）
- 调用 Supervisor Agent 

## 2.2 创建底层工具

我们可以将这个任务的实现拆分为以下几个工具：
- 查询空闲时间 get_available_time_slots
- 创建日程 create_calendar_event
- 发送邮件 send_email

### 2.2.1 查询空闲时间 get_available_time_slots ：

In [ ]:
from langchain.tools import tool
@tool
def get_available_time_slots(
  attendees: list[str],
  date: str, # ISO format: "2024-01-15"
  duration_minutes: int
) -> list[str]:
  """Check calendar availability for given attendees on a specific date."""
  return ["09:00", "14:00", "16:00"]

### 2.2.2 创建日程 create_calendar_event ：

In [ ]:
@tool
def create_calendar_event(
  title: str,
  start_time: str, # ISO format: "2024-01-15T14:00:00"
  end_time: str,  # ISO format: "2024-01-15T15:00:00"
  attendees: list[str], # email addresses
  location: str = ""
) -> str:
  """Create a calendar event. Requires exact ISO datetime format."""
  return f"Event created: {title} from {start_time} to {end_time} with {len(attendees)} attendees"

### 2.2.3 发送邮件 send_email ：

In [ ]:
@tool
def send_email(
  to: list[str],   # email addresses
  subject: str,
  body: str,
  cc: list[str] = []
) -> str:
  """Send an email via email API. Requires properly formatted addresses."""
  return f"Email sent to {', '.join(to)} - Subject: {subject}"

## 2.3 构建子智能体
所谓的子智能体其实是一个只负责「一个垂直任务」的智能体，它自己可以使用工具，但不与用户直接对话。

那我们这里可以根据任务需要将其拆分为两个子智能体：
- 日程管理子智能体 calendar_agent：基于创建日程 create_calendar_event 和查询空闲时间 get_available_time_slots 两个工具实现精准的日常管理工作。
- 邮件发送子智能体 email_agent：基于发送邮件 send_email 工具实现对于指定用户发送特定内容的信息的工作。

和之前构建智能体一样，我们需要先去设置一下其所使用的大模型、工具以及系统提示词（子智能体由于只需要完成指定任务不需要设置记忆）。

### 2.3.1 构建日程管理子智能体 calendar_agent：

In [ ]:
from langchain_community.chat_models import ChatTongyi
from langchain.agents import create_agent
model = ChatTongyi(api_key=os.environ.get("DASHSCOPE_API_KEY"), model="qwen-max")

calendar_agent = create_agent(
    model,
    tools=[create_calendar_event, get_available_time_slots],
    system_prompt=(
    "You are a calendar scheduling assistant. "
    "Parse natural language scheduling requests (e.g., 'next Tuesday at 2pm') "
    "into proper ISO datetime formats. "
    "Use get_available_time_slots to check availability when needed. "
    "Use create_calendar_event to schedule events. "
    "Always confirm what was scheduled in your final response."
    )
)

然后我们可以测试一下其具体的效果（这里使用的是流式输出的方式）：

In [ ]:
query = "Schedule a team meeting next Tuesday at 2pm for 1 hour"

for step in calendar_agent.stream({"messages": [{"role": "user", "content": query}]}):
    for update in step.values():
        for message in update.get("messages", []):
            message.pretty_print()

### 2.3.2 构建发送邮件子智能体 email_agent：


In [ ]:
from langchain_community.chat_models import ChatTongyi
model = ChatTongyi(api_key=os.environ.get("DASHSCOPE_API_KEY"), model="qwen-max")

email_agent = create_agent(
    model,
    tools=[send_email],
    system_prompt=(
    "You are an email assistant. "
    "Compose professional emails based on natural language requests. "
    "Extract recipient information and craft appropriate subject lines and body text. "
    "Use send_email to send the message. "
    "Always confirm what was sent in your final response."
    )
)

然后我们也可以尝试着对其进行调用：

In [ ]:
query = "Send the design team a reminder about reviewing the new mockups"

for step in email_agent.stream({"messages": [{"role": "user", "content": query}]}):
    for update in step.values():
        for message in update.get("messages", []):
            message.pretty_print()

## 2.4 将子代理包装成工具
虽然 calendar_agent 和 email_agent 已经可以独立工作，但 Supervisor_agent 是通过「调用工具」的方式来使用它们的。所以我们还要把子代理包装成新的高层工具，供上层 Supervisor_agent 调度。

所以我们刚刚的两个 agent 会被包装成两个工具：
- schedule_event
- manage_email

当然我们也需要写上对应的函数文档字符串来告诉 Supervisor 在什么时候去使用该工具。

### 2.4.1 构建子代理工具 schedule_event：

In [ ]:
@tool
def schedule_event(request: str) -> str:
  """Schedule calendar events using natural language.

  Use this when the user wants to create, modify, or check calendar appointments.
  Handles date/time parsing, availability checking, and event creation.

  Input: Natural language scheduling request (e.g., 'meeting with design team
  next Tuesday at 2pm')
  """
  result = calendar_agent.invoke({
    "messages": [{"role": "user", "content": request}]
  })
  return result["messages"][-1].text

### 2.4.2 构建子代理工具 manage_email：

In [ ]:
@tool
def manage_email(request: str) -> str:
  """Send emails using natural language.

  Use this when the user wants to send notifications, reminders, or any email
  communication. Handles recipient extraction, subject generation, and email
  composition.

  Input: Natural language email request (e.g., 'send them a reminder about
  the meeting')
  """
  result = email_agent.invoke({
    "messages": [{"role": "user", "content": request}]
  })
  return result["messages"][-1].text

### 2.5 构建 Supervisor Agent（主智能体）

在构建好了 Supervisor 可使用的工具后，我们就可以创建 Supervisor_agent 并准备后续进行调用了，其具体的职责并不是直接使用底层工具，而是：
- 理解用户自然语言请求
- 拆解任务
- 调用合适的子代理工具
- 整合并回复结果

所以根据这个需求，我们可以按以下方式创建 Supervisor_agent :

In [ ]:
from langchain_community.chat_models import ChatTongyi

model = ChatTongyi(api_key=os.environ.get("DASHSCOPE_API_KEY"), model="qwen-max")

supervisor_agent = create_agent(
  model,
  tools=[schedule_event, manage_email],
  system_prompt=(
    "You are a helpful personal assistant. "
    "You can schedule calendar events and send emails. "
    "Break down user requests into appropriate tool calls and coordinate the results. "
    "When a request involves multiple actions, use multiple tools in sequence."
  )
)

## 2.6 调用 Supervisor Agent

假如我们就希望其进行会议的安排：

In [ ]:
query = "Schedule a team standup for tomorrow at 9am"

for step in supervisor_agent.stream(
    {"messages": [{"role": "user", "content": query}]}
):
    for update in step.values():
        for message in update.get("messages", []):
            message.pretty_print()

那这个时候就会单独只调用其中一个工具（schedule_event）来进行完成。

假如我们希望其能够其能够同时安排会议并发邮件提醒，我们也可以进行实现：

In [ ]:
query = (
    "Schedule a meeting with the design team next Tuesday at 2pm for 1 hour, "
    "and send them an email reminder about reviewing the new mockups."
)

for step in supervisor_agent.stream(
    {"messages": [{"role": "user", "content": query}]}
):
    for update in step.values():
        for message in update.get("messages", []):
            message.pretty_print()

可以看到返回的内容为：
- Supervisor_agent 分析了任务后一次性发出了两条工具调用的指令
- 然后子代理 schedule_event 和 manage_email 分别返回了工具调用的结果（非直接工具返回的结果）
- 最后 Supervisor_agent 根据工具调用的信息对问题进行了回复

我们可以将其部署在 LangSmith Studio 上并进行测试（具体可查看 Subagents 文件夹内容）

# 3. Subagents（子智能体）优化

假如我们希望对当前 Tool Calling 模式的 Multi-Agent System 进行优化，我们可以从三个方面入手：
- 增加子智能体能够获取到的信息
- 增加主智能体能够从子智能体中获取到的信息（也就是子智能体返回更多信息）
- 对于一些危险的工具添加人工审核以避免风险
|
## 3.1 增加子智能体获取的信息

在当前的代码里， sub_agent 能看到的信息只是 Supervisor_agent 发送的查询请求。但假如我们希望让子代理看到“上层全部对话历史”或其它状态信息，我们可以通过工具调用时的 ToolRuntime 注入状态：

In [ ]:
from langchain.tools import tool, ToolRuntime

@tool
def schedule_event(request: str, runtime: ToolRuntime) -> str:
  """Schedule calendar events using natural language.

  Use this when the user wants to create, modify, or check calendar appointments.
  Handles date/time parsing, availability checking, and event creation.

  Input: Natural language scheduling request (e.g., 'meeting with design team
  next Tuesday at 2pm')
  """
  # Customize context received by sub-agent
  original_user_message = next(message for message in runtime.state["messages"] if message.type == "human")
  prompt = ("You are assisting with the following user inquiry:\n\n"
      f"{original_user_message.text}\n\n"
      "You are tasked with the following sub-request:\n\n"
      f"{request}")
  result = calendar_agent.invoke({"messages": [{"role": "user", "content": prompt}]})
  return result["messages"][-1].text

所以这个时候 sub_agent 不再只能接收到调用（request）的信息，还能知道部分当前任务的信息并智能化的做出调用的决定。

## 3.2 增加主智能体获取的信息

除了向子代理里注入更多的信息，类似的我们可以让子代理返回更多的信息，比如在 return 的时候并不是仅仅把 result["messages"][-1].text 而是返回一些状态信息：

In [ ]:
@tool
def schedule_event(request: str) -> str:
    """Schedule calendar events using natural language.

    Use this when the user wants to create, modify, or check calendar appointments.
    Handles date/time parsing, availability checking, and event creation.

    Input: Natural language scheduling request (e.g., 'meeting with design team
    next Tuesday at 2pm')
    """
    result = calendar_agent.invoke({"messages": [{"role": "user", "content": request}]})

    # Option 1: Return just the confirmation message
    return result["messages"][-1].text

    # Option 2: Return structured data
    # return json.dumps({
    #     "status": "success",
    #     "event_id": "evt_123",
    #     "summary": result["messages"][-1].text
    # })

通过 json.dumps ，我们可以把特定格式的 json 内容转为字符串的形式然后再返回给大模型。比如上面的内容会得到： ``` "{\"status\": \"success\", \"event_id\": \"evt_123\", \"summary\": \"The event has been scheduled for next Tuesday at 2 PM.\"}" ```。这样Supervisor_agent 就会知道成功运行且运行的 id 是 evt_123 了。

## 3.3 添加人工审核
对于人工审核，可以使用上一章节提到的内置中间件 HumanInTheLoopMiddleware 来实现。这个中间件允许我们能够在“子代理调用工具之前”插入人工判断点。

比如当智能体执行关键操作（如发邮件、创建日程）时，先中断流程，由人类手动“审批 / 编辑 / 拒绝”，再继续执行。

那在我们的系统中，我们也可以添加进该机制，从而实现：
- 确保发出去的邮件不会出错
- 日程安排前想确认时间和人选
- 涉及敏感操作（下单、删库、通知大群）能够提前预判

我们可以给 calendar_agent 添加上人工审核中间件。仅针对 create_calendar_event 工具，而 get_available_time_slots 工具可以自由调用。

In [ ]:
from langchain.agents.middleware import HumanInTheLoopMiddleware

calendar_agent = create_agent(
  model,
  tools=[create_calendar_event, get_available_time_slots],
  system_prompt=("You are a calendar scheduling assistant..."),
  middleware=[ 
    HumanInTheLoopMiddleware( 
      interrupt_on={"create_calendar_event": True}, 
      description_prefix="Calendar event pending approval", 
    ), 
  ], 
)

而对于第二个 email_agent 而言，也可以对 send_email 添加上人工审核。

In [ ]:
email_agent = create_agent(
  model=model,
  tools=[send_email],
  system_prompt=("You are an email assistant... "),
  middleware=[ 
    HumanInTheLoopMiddleware( 
      interrupt_on={"send_email": True}, 
      description_prefix="Outbound email pending approval", 
    ), 
  ], 
)

但是在 HumanInTheLoopMiddleware 中有一个要求，就是必须要在有设置记忆的情况下才能够实现人工审核的操作。所以我们可以为 supervisor_agent 添加 checkpointer 参数：

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

supervisor_agent = create_agent(
    model=model,
    tools=[schedule_event, manage_email],
    system_prompt=("You are a helpful personal assistant. "),
    checkpointer=InMemorySaver()  # 🔥 激活状态保存
)

这样子，在运行过程中的信息就能够保存下来了，那么在传入人工的指令后也可以基于之前的记忆继续恢复了。

对于子代理而言，由于其本身就是为了完成任务而设置的，除非是一些需要基于上下文完成任务的智能体工具，不然是不需要添加记忆进去的。添加了反而可能导致状态不一致或冗余。

当然和前面一样，我们一样可以提出问题，然后调用 supervisor_agent 进行回复：

In [ ]:
from langchain_core.messages import HumanMessage

query = (
    "Schedule a meeting with the design team next Tuesday at 2pm for 1 hour, "
    "and send them an email reminder about reviewing the new mockups."
)

config = {"configurable": {"thread_id": "1"}}

response = supervisor_agent.invoke(
    {"messages": [HumanMessage(content=query)]},
    config=config
)

print(response)

由于存在需要人工介入的情况，因此我们可以参考前面 HumanInTheLoopMiddleware时使用的方法来实现处理。

当需要调用 create_calendar_event 或 send_email 这两个工具时，就会进入到中断的界面，比如我们直接运行代码，会出现：

In [ ]:
from langgraph.types import Command
# 🔁 循环处理多个中断（逐个审批直到流程结束）
while "__interrupt__" in response:
  print("\n🟠 检测到中断，进入人工审核环节...")

  # 🧑 打印用户原始提问
  user_msg = next((m.content for m in response["messages"] if m.type == "human"), "无")
  print(f"\n🧑 用户提问：{user_msg}")

  # 遍历中断对象（可能包含多个 action request）
  for interrupt in response["__interrupt__"]:
    for i, req in enumerate(interrupt.value["action_requests"]):
      print(f"📝 描述：{req.get('description', '无')}")
      print(f"✅ 可选操作：{interrupt.value['review_configs'][i]['allowed_decisions']}")

    # 👤 人工输入决策
    decision_type = input("\n请输入你的决定（approve / edit / reject）：").strip().lower()
    if decision_type not in ["approve", "edit", "reject"]:
      print("⚠️ 无效输入，默认设置为 reject")
      decision_type = "reject"

    decision_payload = {"type": decision_type}

    # ✍️ 如果选择 edit：展示原请求并让用户直接输入完整 JSON
    if decision_type == "edit":
      print("\n🛠️ 当前模型原始调用指令如下：")
      print(json.dumps(req, indent=2, ensure_ascii=False))
      print("\n✍️ 请粘贴你想执行的完整新调用 JSON（例如：）")
      print('{"name": "send_email", "args": {"to": ["x@example.com"], "subject": "测试", "body": "内容"}}')
      try:
        new_raw = input("\n请输入新的调用 JSON：").strip()
        decision_payload["edited_action"] = json.loads(new_raw)
      except Exception as e:
        print(f"❌ JSON 格式错误：{e}")
        print("⚠️ 自动拒绝此调用")
        decision_payload = {"type": "reject", "message": "编辑失败，输入格式错误"}
    # ❌ 如果拒绝，填写理由
    elif decision_type == "reject":
      reason = input("请输入拒绝理由：")
      decision_payload["message"] = reason
    # ✅ 恢复执行（Resume）
    try:
      response = supervisor_agent.invoke(
        Command(resume={interrupt.id: {"decisions": [decision_payload]}}), config=config)
    except Exception as e:
      print(f"❌ 执行恢复时发生错误：{e}")
      break
# ✅ 最终结果输出
print("\n🟢 流程结束，模型最终回复：")
print(response["messages"][-1].content)